In [1]:
import numpy as np
import pandas as pd
import string
import os, sys

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords', quiet=True)

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer, TfidfTransformer
from sklearn.pipeline import Pipeline

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, roc_auc_score

from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
import joblib

# Make backend.utils importable from notebooks/
sys.path.insert(0, os.path.abspath(".."))
from backend.utils import map_labels

In [2]:
# Load the preprocessed dataset (use correct relative path from notebooks/)
df = pd.read_csv("../data/Preprocessed Fake Reviews Detection Dataset.csv")
print("Shape:", df.shape)
df.head()

Shape: (40432, 5)


,Unnamed: 0,category,rating,label,text_
0,0,Home_and_Kitchen_5,5,1,love well made sturdi comfort i love veri pretti
1,1,Home_and_Kitchen_5,5,1,love great upgrad origin i 've mine coupl year
2,2,Home_and_Kitchen_5,5,1,thi pillow save back i love look feel pillow
3,3,Home_and_Kitchen_5,1,1,miss inform use great product price i
4,4,Home_and_Kitchen_5,5,1,veri nice set good qualiti we set two month


In [3]:
# Clean up: drop unnamed index column and NaN rows
if 'Unnamed: 0' in df.columns:
    df.drop('Unnamed: 0', axis=1, inplace=True)

df.dropna(inplace=True)
print("Shape after cleanup:", df.shape)

# Map labels to 'fake' / 'real' for consistency with backend
# This preprocessed CSV uses 1=fake(CG), 0=real(OR)
df['label'] = df['label'].map({1: 'fake', 0: 'real'})
print("Label distribution:")
print(df['label'].value_counts())
df.head()

Shape after cleanup: (40431, 4)
Label distribution:
label
real    20216
fake    20215
Name: count, dtype: int64


,category,rating,label,text_
0,Home_and_Kitchen_5,5,fake,love well made sturdi comfort i love veri pretti
1,Home_and_Kitchen_5,5,fake,love great upgrad origin i 've mine coupl year
2,Home_and_Kitchen_5,5,fake,thi pillow save back i love look feel pillow
3,Home_and_Kitchen_5,1,fake,miss inform use great product price i
4,Home_and_Kitchen_5,5,fake,veri nice set good qualiti we set two month


In [4]:
def text_process(text):

    
    nopunc = [char for char in text if char not in string.punctuation]
    nopunc = ''.join(nopunc)

    
    return [word for word in nopunc.split() if word.lower() not in stopwords.words('english')]

In [5]:
# Train / test split (stratified, reproducible)
X_train, X_test, y_train, y_test = train_test_split(
    df["text_"],
    df["label"],
    test_size=0.2,
    stratify=df["label"],
    random_state=42,
)
print(f"Train: {len(X_train)}  |  Test: {len(X_test)}")

Train: 32344  |  Test: 8087


In [6]:
# Models to compare (balanced class weights where supported, reproducible seeds)
models = {
    "Naive Bayes": MultinomialNB(),
    "Random Forest": RandomForestClassifier(class_weight='balanced', random_state=42, n_estimators=100),
    "Decision Tree": DecisionTreeClassifier(class_weight='balanced', random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=3),
    "SVM": LinearSVC(class_weight='balanced', random_state=42, max_iter=2000),
    "Logistic Regression": LogisticRegression(class_weight='balanced', max_iter=2000, random_state=42),
}

In [7]:
# Train each model inside a CountVectorizer → TfidfTransformer → Classifier pipeline
results = {}
best_pipeline = None
best_acc = 0.0

for name, clf in models.items():

    pipeline = Pipeline([
        ('bow', CountVectorizer(analyzer=text_process, max_features=5000)),
        ('tfidf', TfidfTransformer()),
        ('classifier', clf),
    ])

    pipeline.fit(X_train, y_train)

    predictions = pipeline.predict(X_test)

    accuracy = accuracy_score(y_test, predictions)
    results[name] = accuracy

    # Track best model
    if accuracy > best_acc:
        best_acc = accuracy
        best_pipeline = pipeline
        best_name = name

    print(f"\nMODEL: {name}")
    print(f"Accuracy: {accuracy:.4f}")
    print(classification_report(y_test, predictions, digits=4))


MODEL: Naive Bayes
Accuracy: 0.8426
              precision    recall  f1-score   support

        fake     0.8250    0.8697    0.8467      4043
        real     0.8622    0.8155    0.8382      4044

    accuracy                         0.8426      8087
   macro avg     0.8436    0.8426    0.8425      8087
weighted avg     0.8436    0.8426    0.8425      8087


MODEL: Random Forest
Accuracy: 0.8373
              precision    recall  f1-score   support

        fake     0.8197    0.8647    0.8416      4043
        real     0.8569    0.8098    0.8327      4044

    accuracy                         0.8373      8087
   macro avg     0.8383    0.8373    0.8371      8087
weighted avg     0.8383    0.8373    0.8371      8087


MODEL: Decision Tree
Accuracy: 0.7251
              precision    recall  f1-score   support

        fake     0.7286    0.7173    0.7229      4043
        real     0.7217    0.7329    0.7273      4044

    accuracy                         0.7251      8087
   macro avg 

In [8]:
# Summary table
print("\n===== FINAL MODEL PERFORMANCE =====")
for model_name, acc in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"  {model_name:<25s} : {acc*100:.2f}%")

print(f"\nBest model: {best_name} ({best_acc*100:.2f}%)")


===== FINAL MODEL PERFORMANCE =====
  SVM                       : 86.92%
  Logistic Regression       : 86.41%
  Naive Bayes               : 84.26%
  Random Forest             : 83.73%
  Decision Tree             : 72.51%
  KNN                       : 71.86%

Best model: SVM (86.92%)


In [9]:
# Save the best pipeline to models/
os.makedirs("../models", exist_ok=True)
save_path = "../models/fake_review_model2.pkl"
joblib.dump(best_pipeline, save_path)
print(f"Best pipeline ({best_name}) saved to {save_path}")

Best pipeline (SVM) saved to ../models/fake_review_model2.pkl
